# Step 2b: explaining the posting-timeline models

**Models (4):** 1D CNN · vanilla RNN · LSTM · LSTM autoencoder. Input: 168 hourly post counts (a week), `log1p`-transformed and standardised.

**Two explanation methods per model**

| Model | Method 1 | Method 2 |
|---|---|---|
| CNN | Grad-CAM (on the last convolution block) | Integrated Gradients |
| RNN, LSTM | Integrated Gradients | GradientSHAP |
| LSTM autoencoder | Integrated Gradients | GradientSHAP |

Grad-CAM needs convolutional feature maps, so it only applies to the CNN. All attributions are per **hour** (168 numbers per account). For the classifiers we explain the model's *logit* (the sigmoid of which is the step-1 score); for the autoencoder, its reconstruction error (the anomaly score).

**The answer key.** `truth/sequence_masks.npz` marks the hours that carry the planted pattern: bots burst every 6, 8 or 12 hours, and trolls share two 3-hour windows with their ring. Humans have no planted pattern, so correctness is scored on bots and trolls only.

**Criteria** (definitions in `docs/step2_plan.md`)

* *Correctness:* AUC of |attribution| for telling planted-burst hours from other hours (0.5 = chance), and the share of attribution mass inside the planted hours (compare with the share of hours that are planted).
* *Faithfulness:* set the top-attributed hours to "no posts" and measure the score change versus deleting random hours.
* *Agreement:* the two methods' rank correlation over hours, and overlap of their top-10 hours.
* *Stability:* add small noise to the series, re-explain, correlate. A repeatability control re-runs the same input (matters for GradientSHAP, which samples).
* *Timing probe (diagnostic):* shuffle the hours of every test series. This keeps the set of hourly counts (so total volume and the number of burst hours) but destroys *when* they happen. If a model's AUC barely drops, it reads how much and how bursty the activity is, not its timing. Its explanation can then still point at the burst hours (they are the high-count hours) without the model knowing anything about periodicity.

> Run top to bottom, one cell at a time. `QUICK = True` gives a 1-minute smoke test.

## 0. Setup

In [ ]:
import sys
import time
import warnings
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))  # so `import xai_eval` works from notebooks/

import xai_eval  # noqa: F401  (imports xgboost / lightgbm before torch: macOS OpenMP workaround)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from xai_eval import data as xd
from xai_eval import io as xio
from xai_eval import loaders as L
from xai_eval import metrics as xm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160)
torch.set_num_threads(4)

from captum.attr import GradientShap, IntegratedGradients, LayerAttribution, LayerGradCam
from sklearn.metrics import roc_auc_score

In [ ]:
SEED = 0
QUICK = False

N_PER_KIND = 4 if QUICK else 50
N_STAB_PER_KIND = 3 if QUICK else 20
IG_STEPS = 20 if QUICK else 64        # Integrated Gradients path steps
GSHAP_SAMPLES = 5 if QUICK else 20    # GradientSHAP noisy samples per input
GSHAP_BASELINES = 10 if QUICK else 30 # training series used as the GradientSHAP baseline distribution
GSHAP_STDEV = 0.05                    # noise added to inputs in GradientSHAP (standardised units)
KS = [5, 10, 20, 40]                  # number of hours deleted in the faithfulness test
NOISE_STD = 0.05                      # stability noise, standardised units (the series have overall std 1)
N_RANDOM = 5
FAMILY = "sequences"
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Data and the answer key

In [ ]:
D = xd.load_data(REPO_ROOT)
T = D.X_seq.shape[1]  # 168 hours

S_IDX = D.sample_accounts(per_kind=N_PER_KIND, split="test", seed=SEED)
X_S = D.X_seq[S_IDX]                  # standardised log1p counts of the explained accounts
KIND_S = D.kind[S_IDX]
MASK_S = D.seq_masks[S_IDX]           # ANSWER KEY: planted-burst hours (all False for humans)
STAB_POS = np.concatenate([np.flatnonzero(KIND_S == k)[:N_STAB_PER_KIND] for k in ("human", "bot", "troll")])

# The "no posts" value: what a zero-count hour looks like after log1p + standardisation.
Z0 = float((0.0 - D.seq_mu) / D.seq_sd)
# Baseline distribution for GradientSHAP: random TRAIN series (all kinds)
rng = np.random.default_rng(SEED)
BG_SERIES = D.X_seq[rng.choice(np.flatnonzero(D.is_tr), size=GSHAP_BASELINES, replace=False)]

print(f"{len(S_IDX)} explained accounts; planted share of hours: bots {MASK_S[KIND_S=='bot'].mean():.3f}, trolls {MASK_S[KIND_S=='troll'].mean():.3f}")

In [ ]:
# What the planted pattern looks like (one account of each kind; shaded = planted hours)
fig, axes = plt.subplots(3, 1, figsize=(11, 5.5), sharex=True)
for ax, k in zip(axes, ("human", "bot", "troll")):
    i = np.flatnonzero(KIND_S == k)[0]
    ax.plot(D.counts[S_IDX[i]], color="#333", lw=1)
    for t in np.flatnonzero(MASK_S[i]):
        ax.axvspan(t - 0.5, t + 0.5, color="#E45756", alpha=0.25, lw=0)
    ax.set_ylabel(f"{k}\nposts/hour")
axes[-1].set_xlabel("hour of the week")
plt.tight_layout()
plt.show()

## 2. Reload the step-1 models and check them

In [ ]:
MODELS, CHECKS = {}, []
for name in L.SEQ_MODELS:
    MODELS[name] = L.load_model(name, D)
    CHECKS.append(L.verify_reload(name, D, L.seq_scores(name, MODELS[name], D.X_seq)))
assert all(c["status"] == "OK" for c in CHECKS), "A reloaded model does not match its step-1 predictions"

OUT_FN = {n: L.seq_output_fn(n, MODELS[n]) for n in MODELS}   # the quantity that is explained


def output(name, X):
    """Explained quantity for a batch of standardised series (n, 168): logit, or reconstruction error."""
    with torch.no_grad():
        return OUT_FN[name](L.seq_tensor(name, X)).numpy()


OUT_SD = {n: float(np.std(output(n, D.X_seq[D.is_te]))) for n in MODELS}  # scale for the faithfulness test

## 3. Timing probe: does each model use the timing of posts, or just the volume?

For every test account the 168 hours are randomly permuted. The counts are the same; the order is destroyed, so any periodicity (bots) or shared windows (trolls) disappears. The AUC is computed on the model's step-1 score (for the autoencoder, the reconstruction error).

In [ ]:
rng = np.random.default_rng(SEED)
te = np.flatnonzero(D.is_te)
X_te = D.X_seq[te]
X_shuf = np.stack([row[rng.permutation(T)] for row in X_te])
probe_rows = []
for name in L.SEQ_MODELS:
    auc_orig = roc_auc_score(D.y[te], output(name, X_te))
    auc_shuf = roc_auc_score(D.y[te], output(name, X_shuf))
    probe_rows.append(dict(model=name, auc_original=auc_orig, auc_shuffled_hours=auc_shuf, auc_drop=auc_orig - auc_shuf))
PROBE = pd.DataFrame(probe_rows).set_index("model")
display(PROBE.round(3))

A large `auc_drop` means the model depends on *when* posts happen. An AUC that stays high means the model separates the accounts using the distribution of hourly counts (for example, how many hours are unusually busy) and does not need the timing. That is a legitimate shortcut in this data, because the planted bursts also make a few hours unusually busy. Keep it in mind when reading correctness: an explanation can put its credit on the burst hours and be right about the model, while the model still knows nothing about periodicity.

## 4. Explanation and evaluation code

In [ ]:
METHODS_FOR = {
    "cnn1d": ["Grad-CAM", "Integrated Gradients"],
    "rnn": ["Integrated Gradients", "GradientSHAP"],
    "lstm": ["Integrated Gradients", "GradientSHAP"],
    "autoencoder_lstm": ["Integrated Gradients", "GradientSHAP"],
}


def attribute(name, method, X, seed=SEED):
    """Per-hour attributions (n, 168) of the explained quantity for series X (n, 168). Returns (A, seconds/account)."""
    model, fwd = MODELS[name], OUT_FN[name]
    x = L.seq_tensor(name, X).clone().requires_grad_(True)
    torch.manual_seed(seed)
    t0 = time.time()
    if method == "Grad-CAM":
        cam = LayerGradCam(model, model.features).attribute(x, relu_attributions=True)   # (n, 1, 84)
        A = LayerAttribution.interpolate(cam, (T,), interpolate_mode="linear")           # back to 168 hours
    elif method == "Integrated Gradients":
        A = IntegratedGradients(fwd).attribute(x, baselines=torch.full_like(x, Z0), n_steps=IG_STEPS, internal_batch_size=64)
    elif method == "GradientSHAP":
        base = L.seq_tensor(name, BG_SERIES)
        A = GradientShap(fwd).attribute(x, baselines=base, n_samples=GSHAP_SAMPLES, stdevs=GSHAP_STDEV)
    else:
        raise ValueError(method)
    return A.detach().numpy().reshape(len(X), T), (time.time() - t0) / len(X)


def remove_hours(x, idx):
    x = x.copy()
    x[idx] = Z0
    return x

In [ ]:
SC = xm.Scorecard("timeline")
INST_ROWS, ATTR_STORE = [], {}


def evaluate(name, attrs, secs):
    methods = METHODS_FOR[name]
    predict = lambda rows: output(name, np.vstack(rows))  # noqa: E731
    X_list = list(X_S)
    planted = np.flatnonzero(KIND_S != "human")      # bots and trolls have planted hours
    for m in methods:
        A = attrs[m]
        absA = np.abs(A)
        auc = np.full(len(A), np.nan)
        mass = np.full(len(A), np.nan)
        for i in planted:
            auc[i] = xm.mask_auc(absA[i], MASK_S[i])
            mass[i] = xm.mass_inside(absA[i], MASK_S[i])
        chance_mass = MASK_S[planted].mean()
        SC.add(name, m, "correctness_mask_auc", auc[planted], "chance = 0.5")
        for k in ("bot", "troll"):
            SC.add(name, m, f"correctness_mask_auc_{k}", auc[KIND_S == k], "chance = 0.5")
        SC.add(name, m, "correctness_mass_inside", mass[planted], f"chance = {chance_mass:.3f} (share of hours planted)")
        dele = xm.deletion_gain(predict, X_list, A, remove_hours, KS, OUT_SD[name], n_random=N_RANDOM, seed=SEED)
        dele.assign(model=name, method=m).to_csv(xio.out_dir(D, FAMILY) / f"deletion_{name}_{m.replace(' ', '_')}.csv", index=False)
        faith = xm.mean_gain_per_instance(dele)
        SC.add(name, m, "faithfulness", faith)
        SC.add(name, m, "seconds_per_explanation", secs[m])
        for i in range(len(A)):
            INST_ROWS.append(dict(model=name, method=m, account_id=int(S_IDX[i]), kind=KIND_S[i],
                                  mask_auc=auc[i], mass_inside=mass[i], faithfulness=faith[i]))
    a, b = np.abs(attrs[methods[0]]), np.abs(attrs[methods[1]])
    label = f"{methods[0]} vs {methods[1]}"
    SC.add(name, label, "agreement_spearman", [xm.rank_corr(a[i], b[i]) for i in range(len(a))])
    SC.add(name, label, "agreement_top10", [xm.topk_overlap(a[i], b[i], 10) for i in range(len(a))])


def stability_checks(name, attrs):
    Xs = X_S[STAB_POS]
    noisy = Xs + np.random.default_rng(SEED + 1).normal(0, NOISE_STD, Xs.shape)
    for m in METHODS_FOR[name]:
        base = np.abs(attrs[m][STAB_POS])
        SC.add(name, m, "stability_noise", xm.stability(base, np.abs(attribute(name, m, noisy)[0])))
        SC.add(name, m, "repeatability_seed", xm.stability(base, np.abs(attribute(name, m, Xs, seed=SEED + 100)[0])),
               "1.0 expected for deterministic methods")


def show_examples(name):
    methods = METHODS_FOR[name]
    fig, axes = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True)
    for ax, k in zip(axes, ("human", "bot", "troll")):
        i = int(np.flatnonzero(KIND_S == k)[0])
        for t in np.flatnonzero(MASK_S[i]):
            ax.axvspan(t - 0.5, t + 0.5, color="#E45756", alpha=0.25, lw=0)
        ax.plot(X_S[i], color="#bbb", lw=1, label="series (standardised)")
        for m, col in zip(methods, ("#4C78A8", "#F58518")):
            a = np.abs(ATTR_STORE[name][m][i])
            ax.plot(a / max(a.max(), 1e-12) * np.ptp(X_S[i]) + X_S[i].min(), color=col, lw=1.3, label=f"|{m}| (rescaled)")
        ax.set_ylabel(k)
    axes[0].legend(frameon=False, fontsize=8, ncol=3)
    axes[0].set_title(f"{name}: attribution vs planted hours (red)")
    axes[-1].set_xlabel("hour of the week")
    plt.tight_layout()
    plt.show()


def run_model(name):
    print(f"=== {name} ===")
    attrs, secs = {}, {}
    for m in METHODS_FOR[name]:
        attrs[m], secs[m] = attribute(name, m, X_S)
        print(f"  {m}: {secs[m]:.3f} s per account")
    ATTR_STORE[name] = attrs
    evaluate(name, attrs, secs)
    stability_checks(name, attrs)
    for m, A in attrs.items():
        df = pd.DataFrame(A, columns=[f"h{t}" for t in range(T)])
        df.insert(0, "account_id", S_IDX)
        df.insert(1, "kind", KIND_S)
        xio.save_table(D, FAMILY, f"attributions_{name}_{m.replace(' ', '_')}", df)
    rows = SC.df[(SC.df.model == name) & SC.df.criterion.isin(
        ["correctness_mask_auc", "correctness_mass_inside", "faithfulness", "agreement_spearman", "stability_noise", "repeatability_seed"])]
    display(rows.pivot_table(index="method", columns="criterion", values="value").round(3))
    show_examples(name)

## 5. 1D CNN

Grad-CAM weights the last convolution block's feature maps by how much each channel pushes the output up, then keeps the positive part. The feature maps have half the time resolution (84 steps, after max-pooling), so Grad-CAM is **coarser** than Integrated Gradients; its output is stretched back to 168 hours by interpolation. Expect it to light up regions rather than single hours. Integrated Gradients attributes the output to each input hour by integrating the gradient along a path from the "no posts" series to the real one.

In [ ]:
run_model('cnn1d')

## 6. Vanilla RNN

Integrated Gradients and GradientSHAP. Vanilla RNNs read the sequence step by step with only the *last* hidden state feeding the output, so early hours tend to be forgotten; an honest explanation of such a model may therefore concentrate on the final hours of the week.

In [ ]:
run_model('rnn')

## 7. LSTM

Same two methods as the RNN. The gating lets the LSTM keep information from early in the week, so its attributions should be more spread out than the RNN's.

In [ ]:
run_model('lstm')

## 8. LSTM autoencoder

An unsupervised model: it learns to reconstruct typical weeks, and the reconstruction error is the anomaly score. The explanation says which hours make the week hard to reconstruct (because the explained quantity is the error, both the input and the reconstruction target depend on the hour). Bursts are exactly what a model trained mostly on circadian human weeks cannot reproduce, so this is also a test of whether anomaly-based explanations point to the planted pattern.

In [ ]:
run_model('autoencoder_lstm')

## 9. Scorecard for the timeline group

In [ ]:
tbl = SC.table()
order = ["correctness_mask_auc", "correctness_mask_auc_bot", "correctness_mask_auc_troll", "correctness_mass_inside",
         "faithfulness", "agreement_spearman", "agreement_top10", "stability_noise", "repeatability_seed", "seconds_per_explanation"]
tbl = tbl[[c for c in order if c in tbl.columns]]
display(tbl.round(3))
print("Timing probe (AUC when the hours of each test series are shuffled):")
display(PROBE.round(3))

In [ ]:
df = SC.df
crits = ["correctness_mask_auc", "correctness_mass_inside", "faithfulness", "stability_noise"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharey=True)
rows = [(m, meth) for m in L.SEQ_MODELS for meth in METHODS_FOR[m]]
for ax, c in zip(axes, crits):
    d = df[df.criterion == c].set_index(["model", "method"]).reindex(rows)
    y = np.arange(len(rows))
    ax.barh(y, d["value"], 0.6, color="#4C78A8")
    ax.errorbar(d["value"], y, xerr=[(d["value"] - d["ci_low"]).clip(lower=0), (d["ci_high"] - d["value"]).clip(lower=0)],
                fmt="none", ecolor="black", lw=0.8, capsize=2)
    ax.set_title(c.replace("_", " "))
    ax.axvline(0.5 if c == "correctness_mask_auc" else 0, color="gray", lw=0.8)
axes[0].set_yticks(np.arange(len(rows)))
axes[0].set_yticklabels([f"{m} / {meth}" for m, meth in rows], fontsize=8)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Average |attribution| per hour, split by kind: does the credit follow the planted hours?
fig, axes = plt.subplots(len(L.SEQ_MODELS), 3, figsize=(14, 2.2 * len(L.SEQ_MODELS)), sharex=True)
for r, name in enumerate(L.SEQ_MODELS):
    m = METHODS_FOR[name][0]
    A = np.abs(ATTR_STORE[name][m])
    for c, k in enumerate(("human", "bot", "troll")):
        sel = KIND_S == k
        axes[r, c].plot(A[sel].mean(0) / max(A.mean(), 1e-12), color="#4C78A8")
        if k != "human":
            axes[r, c].fill_between(np.arange(T), 0, MASK_S[sel].mean(0) * axes[r, c].get_ylim()[1], color="#E45756", alpha=0.2, lw=0)
        axes[r, c].set_title(f"{name} / {m} / {k}", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
xio.save_scorecard(D, FAMILY, SC, reload_checks=CHECKS, meta=dict(
    seed=SEED, quick=QUICK, n_per_kind=N_PER_KIND, ig_steps=IG_STEPS, gshap_samples=GSHAP_SAMPLES,
    gshap_baselines=GSHAP_BASELINES, ks=KS, noise_std=NOISE_STD, z0=Z0, methods=METHODS_FOR))
xio.save_table(D, FAMILY, "timing_probe", PROBE.reset_index())
pd.DataFrame(INST_ROWS).to_csv(xio.out_dir(D, FAMILY) / "per_instance_scores.csv", index=False)
tbl.round(4).to_csv(xio.out_dir(D, FAMILY) / "scorecard_wide.csv")

## 10. What to take from this notebook

1. Does any model's explanation find the planted bursts (mask AUC well above 0.5), and does that depend on whether the model relies on timing (timing probe)?
2. Is Grad-CAM, being coarser, penalised on mask AUC compared with Integrated Gradients, or does it localise the bursts just as well?
3. Do the recurrent models' explanations concentrate on the last hours of the week?
4. Do the two methods agree more for the CNN than for the recurrent models?
5. How noisy is GradientSHAP between runs (`repeatability_seed`)?